In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.linear_model import LinearRegression

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

In [ ]:
customers = pd.read_csv("../data/customers_clean.csv")
history = pd.read_csv("../data/history_clean.csv")
ref_train = pd.read_csv("../data/referance_data.csv")
ref_test = pd.read_csv("../data/referance_data_test.csv")

In [ ]:
# Tarihler
history["date"] = pd.to_datetime(history["date"])
ref_train["ref_date"] = pd.to_datetime(ref_train["ref_date"])
ref_test["ref_date"] = pd.to_datetime(ref_test["ref_date"])

print("✅ Temiz veriler yüklendi ve tarih formatları hazır.")

In [ ]:
# ==========================================================
# DEMOGRAFİK FEATURELAR
# ==========================================================

# Tenure'ı yıllık kategoriye çevir
customers["NEW_TENURE_GROUP"] = pd.cut(
    customers["tenure"],
    bins=[0, 12, 36, 60, 120, customers["tenure"].max()],
    labels=["0-1 Year", "1-3 Years", "3-5 Years", "5-10 Years", "10+ Years"]
)

# Yaş grubu
customers["NEW_AGE_GROUP"] = pd.cut(
    customers["age"],
    bins=[0, 25, 40, 55, 70, 120],
    labels=["Young", "Adult", "Mid-Age", "Senior", "Old"]
)


In [ ]:
# Toplam harcama ve işlem sayısı
history["NEW_TOTAL_TX_AMT"] = history["cc_transaction_all_amt"] + history["mobile_eft_all_amt"]
history["NEW_TOTAL_TX_CNT"] = history["cc_transaction_all_cnt"] + history["mobile_eft_all_cnt"]

# EFT/CC oranı
history["NEW_EFT_TO_CC_RATIO"] = history["mobile_eft_all_amt"] / (history["cc_transaction_all_amt"] + 1)

# Aktivite flag
history["NEW_ACTIVITY_FLAG"] = np.where(
    (history["cc_transaction_all_cnt"] == 0) &
    (history["mobile_eft_all_cnt"] == 0),
    0, 1
)


In [ ]:
def fast_create_features(history_df, reference_df, months=6):
    print("⏳ Özellikler hesaplanıyor...")

    # 1. Tarihe göre sırala
    history_df = history_df.sort_values(["cust_id", "date"]).copy()

    # 2. Rolling işlemi için index'i date yap
    grouped = (
        history_df
        .set_index("date")   # 🔥 ekledik
        .groupby("cust_id")[["NEW_TOTAL_TX_AMT", "NEW_TOTAL_TX_CNT", "active_product_category_nbr",
                             "NEW_EFT_TO_CC_RATIO", "NEW_ACTIVITY_FLAG"]]
        .rolling(window=months, min_periods=1)
        .agg({
            "NEW_TOTAL_TX_AMT": ["mean", "sum", "var"],
            "NEW_TOTAL_TX_CNT": ["mean", "sum"],
            "active_product_category_nbr": "mean",
            "NEW_EFT_TO_CC_RATIO": "mean",
            "NEW_ACTIVITY_FLAG": "mean"
        })
    )

    # 3. Kolon isimlerini düzleştir
    grouped.columns = ['_'.join(col).strip() for col in grouped.columns.values]
    grouped = grouped.reset_index()   # burada hem cust_id hem date geri gelir

    # 4. Reference data ile birleştir
    merged = pd.merge(
        reference_df[["cust_id", "ref_date"]],
        grouped,
        on="cust_id",
        how="left"
    )

    # 🔥 5. Tarih karşılaştırması (artık date datetime!)
    merged = merged[merged["date"] < merged["ref_date"]]

    # 6. Her müşteri-ref_date için son kayıt (yani en güncel 6 aylık özet)
    merged = merged.groupby(["cust_id", "ref_date"]).last().reset_index()

    # 🔹 Ek: Ortalama işlem tutarı (sum/sum)
    merged["avg_tx_amt_last6m"] = merged["NEW_TOTAL_TX_AMT_sum"] / (merged["NEW_TOTAL_TX_CNT_sum"] + 1)

    print(f"✅ Özellik üretimi tamamlandı → {merged.shape[0]} müşteri için özet hazır.")
    return merged


In [ ]:
train_features = fast_create_features(history, ref_train, months=6)
test_features = fast_create_features(history, ref_test, months=6)


In [ ]:
# ==========================================================
# 02 — FEATURE SONRASI TEMİZLİK + TİP AYARLARI
# ==========================================================

# 1) Türev feature’larda NaN/inf temizliği
num_fill_0 = [
    "NEW_TOTAL_TX_AMT_var",         # varyans tek gözlemde NaN olabilir
    "NEW_EFT_TO_CC_RATIO_mean",     # bölen 0 olduğunda NaN/inf olabilir
    "NEW_ACTIVITY_FLAG_mean",       # sürpriz NaN'lar
    "avg_tx_amt_last6m"             # bölen 0 olduğunda NaN olabilir
]

for df_ in [train_features, test_features]:
    df_[num_fill_0] = df_[num_fill_0].replace([np.inf, -np.inf], np.nan).fillna(0)

# 2) Demografik kategorileri "category" dtype yap (LGBM/CatBoost dostu)
cat_cols = ["gender", "province", "religion", "work_type", "work_sector",
            "NEW_TENURE_GROUP", "NEW_AGE_GROUP"]

for df_ in [customers]:
    for c in cat_cols:
        df_[c] = df_[c].astype("category")

# 3) Birleştir (sen zaten yaptın; burada sadece emin olmak için)
train_df = (
    train_features
    .merge(customers, on="cust_id", how="left")
    .merge(ref_train[["cust_id", "churn"]], on="cust_id", how="left")
)

test_df = (
    test_features
    .merge(customers, on="cust_id", how="left")
)

# 4) Son kontrol ve kaydetme
print(train_df.isna().sum().sort_values(ascending=False).head(10))
train_df.to_csv("../data/train_features_fast.csv", index=False)
test_df.to_csv("../data/test_features_fast.csv", index=False)
print("✅ Temizlenmiş feature set kaydedildi.")


In [ ]:
print(train_df.shape)
print(train_df.head())
print(train_df.isnull().sum().sort_values(ascending=False).head(10))
